# Segment Anything 모델(SAM)

이 노트북은 [Ultralytics Python 패키지](https://pypi.org/project/ultralytics/)로 [Segment Anything 3(SAM 3)](https://docs.ultralytics.com/models/sam-3), [SAM 2](https://docs.ultralytics.com/models/sam-2) 또는 기존 SAM 모델을 사용하기 위한 입문 자료입니다.


## SAM, SAM 2 및 SAM 3 소개

SAM은 이미지 분할을 위한 새로운 모델, 과제 및 데이터셋을 제시한 획기적인 프로젝트입니다. 발전된 설계 덕분에 사전 지식 없이 새로운 이미지 분포와 과제에 적응할 수 있으며, 이러한 기능을 제로샷 전이(zero-shot transfer)라고 합니다.

- SAM은 엄선된 이미지 1,100만 장에 걸쳐 10억 개 이상의 마스크를 포함하는 대규모 SA-1B 데이터셋으로 학습되었습니다. 뛰어난 제로샷 성능을 보여 주었으며, 많은 경우 기존의 완전 지도 학습 결과를 능가했습니다.

[SAM 2](https://docs.ultralytics.com/models/sam-2#key-features)는 Meta의 Segment Anything 모델(SAM)의 후속 모델로, 이미지와 동영상 모두에서 포괄적인 객체 분할을 수행하도록 설계되었습니다. 실시간 처리와 제로샷 일반화를 지원하는 통합된 프롬프트 기반 모델 구조를 통해 복잡한 시각 데이터를 효과적으로 처리합니다.

[SAM 3](https://docs.ultralytics.com/models/sam-3)는 프롬프트 기반 개념 분할(Promptable Concept Segmentation, PCS)을 위한 Meta의 최신 파운데이션 모델입니다. 프롬프트 하나당 객체 하나를 분할하는 이전 SAM 버전과 달리, SAM 3는 텍스트 프롬프트, 이미지 예시 또는 둘 모두로 표현된 시각적 개념에 해당하는 **모든 객체 인스턴스**를 찾아 분할할 수 있습니다. 개념 분할에서는 기존 시스템 대비 2배의 성능 향상을 달성하면서, SAM 2의 시각적 프롬프트(점, 박스, 마스크)와도 완전히 호환됩니다.


![SAM 추론 결과](https://github.com/user-attachments/assets/193d69bc-ae6a-43c4-98d6-9d7a4e86aa99)

## 환경 설정

pip install로 `ultralytics`와 [필요한 패키지](https://github.com/ultralytics/ultralytics/blob/main/pyproject.toml)를 설치하고, 소프트웨어와 하드웨어 환경을 확인합니다.

[![PyPI - 버전](https://img.shields.io/pypi/v/ultralytics?logo=pypi&logoColor=white)](https://pypi.org/project/ultralytics/) [![다운로드 수](https://static.pepy.tech/badge/ultralytics)](https://clickpy.clickhouse.com/dashboard/ultralytics) [![PyPI - Python 버전](https://img.shields.io/pypi/pyversions/ultralytics?logo=python&logoColor=gold)](https://pypi.org/project/ultralytics/)


In [ ]:
!uv pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.3.79 🚀 Python-3.11.11 torch-2.5.1+cu124 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 33.2/112.6 GB disk)


## 전체 내용 분할하기

SAM, SAM 2 및 SAM 3 모델을 사용하면 별도의 프롬프트 없이 이미지나 동영상의 전체 내용을 [분할](https://docs.ultralytics.com/tasks/segment)할 수 있습니다.


In [ ]:
from ultralytics import SAM

# 모델 불러오기
# 모델별 가중치: SAM=sam_b.pt, SAM 2=sam2_b.pt, SAM 2.1=sam2.1_b.pt (기본값이며 바로 실행 가능)
# model="sam3.pt"로 설정하면 SAM 3를 사용할 수 있습니다. SAM 3 가중치는 아래 주소에서 접근 승인을 받은 뒤 다운로드해야 합니다.
# https://huggingface.co/facebook/sam3 — 먼저 sam3.pt를 작업 디렉터리에 넣으세요.
model = SAM("sam2.1_b.pt")

model.info()  # 모델 정보 표시 (선택 사항)

# 추론 수행 (이미지 또는 동영상)
results = model("https://ultralytics.com/images/bus.jpg")  # 이미지
# results = model("https://youtu.be/LNwODJXcvt4")  # 동영상 파일

results[0].show()  # 결과 표시

100%|██████████| 154M/154M [00:00<00:00, 325MB/s]


Model summary: 403 layers, 80,850,178 parameters, 80,850,178 gradients



100%|██████████| 134k/134k [00:00<00:00, 8.17MB/s]


image 1/1 /content/bus.jpg: 1024x1024 1 0, 1 1, 1 2, 1 3, 1 4, 1 5, 1 6, 1 7, 1 8, 1 9, 1 10, 1 11, 1 12, 1 13, 1 14, 1 15, 1 16, 1 17, 1 18, 1 19, 1 20, 1 21, 8291.7ms
Speed: 9.2ms preprocess, 8291.7ms inference, 1.8ms postprocess per image at shape (1, 3, 1024, 1024)


![전체 내용 분할 결과](https://github.com/user-attachments/assets/5d00c0e6-42c3-4f23-9975-1340d9b866f5)

## 원하는 객체 분할하기

바운딩 박스나 점 등 다양한 프롬프트를 사용하여 이미지나 동영상에서 특정 객체를 [분할](https://docs.ultralytics.com/tasks/segment)할 수 있습니다.


### 바운딩 박스 프롬프트

`bbox_prompt`는 모델이 이미지 내의 특정 객체를 분할하도록 안내하는 바운딩 박스 입력을 뜻합니다. 아래 예제에서는 바운딩 박스 좌표를 제공하여 버스만 분할합니다.


In [ ]:
from ultralytics import SAM

# 모델 불러오기 (SAM 3 접근 권한이 있다면 "sam3.pt"로 변경)
model = SAM("sam2.1_b.pt")

# 바운딩 박스 프롬프트로 추론 수행 (버스 영역의 바운딩 박스 좌표를 제공하여
# 이미지 전체에서 버스만 분할되도록 합니다.)
results = model("https://ultralytics.com/images/bus.jpg",
                bboxes=[3.8328723907470703, 229.35601806640625,
                        796.2098999023438, 728.4313354492188])

results[0].show()  # 결과 표시


Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
image 1/1 /content/bus.jpg: 1024x1024 1 0, 334.1ms
Speed: 7.8ms preprocess, 334.1ms inference, 0.7ms postprocess per image at shape (1, 3, 1024, 1024)


![바운딩 박스 프롬프트를 이용한 객체 분할 결과](https://github.com/user-attachments/assets/4833d1fe-7990-4829-83d9-dc6e43b9d08c)

### 점 프롬프트

`point_prompt`는 Segment Anything 모델(SAM)이 이미지 내의 객체를 분할하도록 안내하는 특정 점의 좌표 (x, y) 입력을 뜻합니다. 바운딩 박스를 제공하는 대신 객체 위의 한 점을 선택하여 객체를 지정할 수 있으며, SAM은 해당 점 주변의 분할 마스크를 생성합니다.


In [ ]:
from ultralytics import SAM

# 모델 불러오기 (SAM 3 접근 권한이 있다면 "sam3.pt"로 변경)
model = SAM("sam2.1_b.pt")

# 점 프롬프트로 추론 수행 (사람 영역의 점 좌표를 제공하여
# 이미지 전체에서 해당 사람만 분할되도록 합니다.)
results = model("https://ultralytics.com/images/bus.jpg",
                points=[34, 714])

results[0].show()  # 결과 표시


Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
image 1/1 /content/bus.jpg: 1024x1024 1 0, 312.3ms
Speed: 6.4ms preprocess, 312.3ms inference, 0.7ms postprocess per image at shape (1, 3, 1024, 1024)


![점 프롬프트를 이용한 객체 분할 결과](https://github.com/user-attachments/assets/fc218266-1e1a-4e31-a0e3-de4de55cb13c)

### 여러 점을 이용한 프롬프트

여러 개의 `point_prompt` 입력은 Segment Anything 모델(SAM)이 이미지 내의 여러 객체를 분할하도록 안내하는 프롬프트 역할을 하는 점들의 좌표 (x, y)를 뜻합니다.


In [ ]:
from ultralytics import SAM

# 모델 불러오기 (SAM 3 접근 권한이 있다면 "sam3.pt"로 변경)
model = SAM("sam2.1_b.pt")

# 여러 점 프롬프트로 추론 수행 (사람 영역의 점 좌표들을 제공하여
# 이미지 전체에서 해당 사람만 분할되도록 합니다.)
results = model("https://ultralytics.com/images/bus.jpg",
                points=[[34, 714], [283, 634]])

results[0].show()  # 결과 표시


Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
image 1/1 /content/bus.jpg: 1024x1024 1 0, 1 1, 315.8ms
Speed: 7.4ms preprocess, 315.8ms inference, 0.7ms postprocess per image at shape (1, 3, 1024, 1024)


![여러 점 프롬프트를 이용한 객체 분할 결과](https://github.com/user-attachments/assets/fd7f7d78-f79b-401f-ad03-e129e8dc89dd)

## Segment Anything 모델을 이용한 자동 어노테이션

[자동 어노테이션](https://docs.ultralytics.com/reference/data/annotator#ultralytics.data.annotator.auto_annotate)은 SAM의 핵심 기능으로, 사전 학습된 검출 모델을 사용하여 분할 데이터셋을 만들 수 있게 합니다. 대규모 이미지 집합에 빠르고 정확하게 정답 정보를 부여하여 어노테이션 과정을 간소화하고, 많은 수작업이 필요한 어노테이션 작업을 줄여 줍니다.

- 아래 예제에서는 Ultralytics에서 제공하는 이미지 두 장으로 자동 어노테이션을 수행하지만, 직접 준비한 이미지를 사용할 수도 있습니다. 폴더를 만들고 자동으로 어노테이션할 이미지를 넣은 뒤, 해당 폴더 경로를 `auto_annotate` 함수의 `data` 인수로 전달하면 됩니다.


In [ ]:
from ultralytics.data.annotator import auto_annotate

# Ultralytics 예제 데이터셋 사용 (직접 준비한 이미지를 사용할 수도 있습니다.)
from ultralytics.utils.downloads import safe_download
images = ["bus.jpg", "zidane.jpg"]
for img in images:
  path = safe_download(f"https://ultralytics.com/assets/{img}", dir="assets")

# 어노테이션을 Ultralytics YOLO 분할 형식으로 반환합니다.
# 출력 디렉터리: assets_auto_annotate_labels
# SAM 3 접근 권한이 있다면 sam_model="sam3.pt"로 변경하여 개념 기반 자동 어노테이션을 수행하세요.
auto_annotate(data="assets",
              det_model="yolo26x.pt",
              sam_model="sam_b.pt")

100%|██████████| 134k/134k [00:00<00:00, 6.56MB/s]


100%|██████████| 49.2k/49.2k [00:00<00:00, 8.62MB/s]


100%|██████████| 109M/109M [00:00<00:00, 342MB/s]


100%|██████████| 358M/358M [00:01<00:00, 200MB/s]



image 1/2 /content/assets/bus.jpg: 640x480 4 persons, 1 bus, 71.2ms
image 2/2 /content/assets/zidane.jpg: 384x640 2 persons, 3 ties, 33.6ms
Speed: 2.8ms preprocess, 52.4ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)


## 추가 자료  

🔹 Segment Anything 3 문서: [📖 문서 읽기](https://docs.ultralytics.com/models/sam-3)  
🔹 Segment Anything 2 문서: [📖 문서 읽기](https://docs.ultralytics.com/models/sam-2)  
🔹 SAM2 블로그: [📝 활용 사례 살펴보기](https://www.ultralytics.com/blog/applications-of-meta-ai-segment-anything-model-2-sam-2)
